# LocalScript — QLoRA Fine-Tuning

Fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on Octapi Lua generation tasks.

**Runtime:** GPU — T4 (free) or A100 (faster).  
**Estimated time:** ~45 min on T4 with 300 examples, 3 epochs.

**Workflow:**
1. Setup — install deps, clone repo
2. Upload dataset — `data/train.jsonl` from your local machine
3. Train — QLoRA fine-tune
4. Export — merge + convert to GGUF
5. Download — save GGUF to your local machine
6. Register — `ollama create localscript -f Modelfile`

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Unsloth provides optimised QLoRA training with 2-5x speedup.
# This cell takes ~3 minutes on first run.
!pip install unsloth -q
!pip install trl transformers datasets peft bitsandbytes -q
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Verify GPU ─────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No GPU found — switch runtime to GPU!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu}  |  VRAM: {vram:.1f} GB')

In [ ]:
# ── Cell 3: Upload dataset ─────────────────────────────────────────────────────
# Upload data/train.jsonl from your local machine.
# Run `make merge-data` locally first to produce it.
from google.colab import files
import os

DATASET_PATH = 'train.jsonl'

if not os.path.exists(DATASET_PATH):
    print('Upload your data/train.jsonl file:')
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        os.rename(fname, DATASET_PATH)
        print(f'Saved as {DATASET_PATH}')

# Count examples
with open(DATASET_PATH) as f:
    n = sum(1 for line in f if line.strip())
print(f'Dataset: {n} examples')

In [ ]:
# ── Cell 4: Configuration ──────────────────────────────────────────────────────
BASE_MODEL   = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ_LEN  = 1024
OUTPUT_DIR   = 'checkpoints'
LORA_R       = 16
EPOCHS       = 3
BATCH_SIZE   = 2
GRAD_ACCUM   = 4
LR           = 2e-4

print(f'Model:   {BASE_MODEL}')
print(f'Epochs:  {EPOCHS}  |  Batch: {BATCH_SIZE}  |  Grad accum: {GRAD_ACCUM}')
print(f'LoRA r:  {LORA_R}  |  LR: {LR}')

In [ ]:
# ── Cell 5: Load model in 4-bit ────────────────────────────────────────────────
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template='qwen-2.5')
print('Model loaded.')

In [ ]:
# ── Cell 6: Attach LoRA adapter ────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_R,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapter attached.')

In [ ]:
# ── Cell 7: Format dataset ─────────────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset('json', data_files=DATASET_PATH, split='train')

def _format(batch):
    texts = []
    inputs = batch.get('input', [''] * len(batch['instruction']))
    for inst, inp, out in zip(batch['instruction'], inputs, batch['output']):
        user_content = f'{inp}\n\n{inst}' if inp else inst
        messages = [
            {'role': 'user',      'content': user_content},
            {'role': 'assistant', 'content': out},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {'text': texts}

dataset = raw.map(_format, batched=True, remove_columns=raw.column_names)
print(f'Formatted {len(dataset)} examples.')
print('Sample:')
print(dataset[0]['text'][:300], '...')

In [ ]:
# ── Cell 8: Train ──────────────────────────────────────────────────────────────
import os
from trl import SFTTrainer
from transformers import TrainingArguments

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        learning_rate=LR,
        fp16=True,
        logging_steps=10,
        save_strategy='epoch',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        report_to='none',
        seed=42,
    ),
)

print('Training started...')
trainer.train()

model.save_pretrained(f'{OUTPUT_DIR}/final')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/final')
print(f'Checkpoint saved to {OUTPUT_DIR}/final')

In [ ]:
# ── Cell 9: Merge adapter into base weights ────────────────────────────────────
MERGED_DIR = 'merged'

model, tokenizer = FastLanguageModel.from_pretrained(
    f'{OUTPUT_DIR}/final',
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    dtype=None,
)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'Merged weights saved to {MERGED_DIR}')

In [ ]:
# ── Cell 10: Convert to GGUF Q4_K_M ───────────────────────────────────────────
# Clone and build llama.cpp (takes ~2 minutes).
import subprocess

GGUF_PATH = 'localscript-q4_k_m.gguf'

if not __import__('os').path.exists('llama.cpp'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggerganov/llama.cpp.git'], check=True)
    subprocess.run(['pip', 'install', '-r', 'llama.cpp/requirements.txt', '-q'], check=True)

import sys
subprocess.run([
    sys.executable, 'llama.cpp/convert_hf_to_gguf.py',
    MERGED_DIR,
    '--outtype', 'q4_k_m',
    '--outfile', GGUF_PATH,
], check=True)

size_mb = __import__('os').path.getsize(GGUF_PATH) / 1e6
print(f'GGUF ready: {GGUF_PATH} ({size_mb:.0f} MB)')

In [ ]:
# ── Cell 11: Download GGUF to your local machine ───────────────────────────────
from google.colab import files
print(f'Downloading {GGUF_PATH} ...')
files.download(GGUF_PATH)

## Next Steps (on your local machine)

1. Copy the downloaded GGUF to `training/localscript-q4_k_m.gguf` in the repo.

2. Register the model with Ollama:
   ```bash
   cd training
   ollama create localscript -f Modelfile
   ollama list   # confirm localscript:latest appears
   ```

3. Run the smoke test:
   ```bash
   make eval
   ```

4. If quality is insufficient, generate more data and repeat:
   ```bash
   make generate-data ROLE=generator COUNT=100   # manual
   # or
   make auto-data ROLE=generator COUNT=100       # OpenAI API
   make merge-data
   # re-upload train.jsonl to Colab and re-run from Cell 7
   ```